# Smart-Energy-Intelligence-Platform
## Milestone 4.5: Google Colab GPU Environment Setup & Diagnostics
**Research Project:** *Cross-Country Generalization of NILM Models for Indian Residential Energy Consumption*

---

> [!IMPORTANT]
> **Research Lock Notice:**  
> This notebook is for **environment configuration, dependency verification, and GPU sanity checks only**.
> Full M4.5 training on the complete $1,144,163$-window REFIT dataset is locked by default and cannot be accidentally executed.

### 1. Compute Environment & GPU Diagnostics
Inspect system hardware, PyTorch version, CUDA runtime availability, GPU model, and VRAM memory.

In [ ]:
import os
import sys
import platform
import torch

print('=' * 80)
print('GOOGLE COLAB COMPUTE ENVIRONMENT DIAGNOSTICS')
print('=' * 80)
print(f'  OS & Platform   : {platform.system()} {platform.release()} ({platform.machine()})')
print(f'  Python Version  : {sys.version.split()[0]}')
print(f'  PyTorch Version : {torch.__version__}')
print(f'  CUDA Available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'  CUDA Version    : {torch.version.cuda}')
    print(f'  GPU Device Count: {torch.cuda.device_count()}')
    print(f'  Active GPU Name : {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    total_vram_gb = props.total_memory / (1024 ** 3)
    print(f'  Total GPU VRAM  : {total_vram_gb:.2f} GB')
    print(f'  Compute Capab.  : {props.major}.{props.minor}')
else:
    print('  [WARNING] CUDA is NOT available. Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU / A100 GPU.')
print('=' * 80)

### 2. Repository Setup & Branch Verification
Clone or verify the `Smart-Energy-Intelligence-Platform` repository on branch `ML` with the approved CUDA migration commit.

In [ ]:
import os
from pathlib import Path

REPO_NAME = 'Smart-Energy-Intelligence-Platform'
REPO_URL = 'https://github.com/Abhishek882772/Smart-Energy-Intelligence-Platform.git'
BRANCH = 'ML'

# Check if working directory is inside cloned repository
if not Path(REPO_NAME).exists() and not Path('ML').exists():
    print(f'Cloning repository from {REPO_URL} (Branch: {BRANCH})...')
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}
elif Path(REPO_NAME).exists():
    %cd {REPO_NAME}

print('\nChecking current Git status and commit history:')
!git status
!git log --oneline -3

### 3. Dependency Verification
Verify all required project packages (`torch`, `numpy`, `pandas`, `matplotlib`, `pyyaml`, `scikit-learn`).

In [ ]:
!pip install -q pyyaml matplotlib pandas scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import sklearn

print('Dependency Verification:')
print(f'  - NumPy        : {np.__version__}')
print(f'  - Pandas       : {pd.__version__}')
print(f'  - PyYAML       : {yaml.__version__}')
print(f'  - Scikit-Learn : {sklearn.__version__}')
print(f'  - PyTorch      : {torch.__version__}')
print('  [SUCCESS] All required dependencies verified.')

### 4. Project Import & Path Setup
Add the repository root to `sys.path` and verify that all core ML modules can be imported cleanly.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Repository Root Path: {repo_root}')

# Verify imports from ML.src
from ML.src.models.seq2point import MultiOutputSeq2Point
from ML.src.models.dataset import REFITShardDataset
from ML.src.losses.masked_multitask_loss import MaskedMultiTaskLoss
from ML.src.training import (
    Trainer,
    Evaluator,
    CheckpointManager,
    TrainingHistory,
    get_device,
    get_dataloader_kwargs,
    log_environment_info,
)

print('  [SUCCESS] All ML.src models, losses, and training modules imported successfully.')

### 5. Materialized REFIT Dataset Setup
Mount Google Drive or extract the already-materialized `.npz` dataset shards into `ML/data/processed/REFIT/`.

> [!WARNING]
> **DO NOT** download or regenerate the 6+ GB raw REFIT dataset.  
> Use the existing materialized shards ($159.0\text{ MB}$ compressed NPZ files).

In [ ]:
# OPTION A: Mount Google Drive if dataset is archived in Drive
# from google.colab import drive
# drive.mount('/content/drive')

# OPTION B: Copy / Unzip materialized REFIT dataset into ML/data/processed/REFIT
processed_dir = repo_root / 'ML/data/processed/REFIT'
processed_dir.mkdir(parents=True, exist_ok=True)

print(f'Target Processed Data Directory: {processed_dir}')
print('Place your materialized NPZ shards into train/, validation/, test/, and domain_shift/ under this path.')

### 6. Dataset Integrity & Shard Verification
Verify that all 13 materialized household shards across the 4 disjoint partitions exist and are ready for ingestion.

In [ ]:
partitions = {
    'train': [2, 3, 5, 7, 9],
    'validation': [1, 8, 15],
    'test': [6, 10, 11, 20],
    'domain_shift': [21],
}

all_shards_found = True
total_windows = 0

print('Materialized REFIT Shard Verification:')
for split_name, houses in partitions.items():
    split_dir = processed_dir / split_name
    print(f'\nPartition: {split_name.upper()} ({split_dir})')
    if not split_dir.exists():
        print(f'  [MISSING] Directory does not exist: {split_dir}')
        all_shards_found = False
        continue
    for h in houses:
        shard_file = split_dir / f'house_{h}.npz'
        if shard_file.exists():
            data = np.load(shard_file)
            w_count = len(data['X'])
            total_windows += w_count
            print(f'  - House {h:2d} Shard: {shard_file.name:<15} -> {w_count:>9,} windows ({data["X"].shape})')
        else:
            print(f'  - House {h:2d} Shard: [NOT FOUND] {shard_file.name}')
            all_shards_found = False

if all_shards_found:
    print(f'\n[PASS] All 13 household shards verified! Total sliding windows: {total_windows:,}')
else:
    print('\n[NOTE] Some materialized shards are not yet present in the local directory. Please populate ML/data/processed/REFIT before running training.')

### 7. Unit Test Suite Execution
Execute the project's automated test suites to verify architecture math, loss masking, dataset integration, and device-safe training modules.

In [ ]:
print('1. Running Model & Loss Unit Tests:')
!python ML/tests/test_seq2point_model.py

print('\n2. Running Training Framework Unit Tests:')
!python ML/tests/test_training_framework.py

if (processed_dir / 'train/house_2.npz').exists():
    print('\n3. Running Real REFIT Dataset Integration Test:')
    !python ML/tests/test_dataset_integration.py
else:
    print('\n3. Skipping test_dataset_integration.py (house_2.npz not yet populated).')

### 8. GPU Forward/Backward Smoke Test
Verify device placement, tensor operations, forward pass, masked multi-task loss, and backward gradient flow on GPU using a synthetic mini-batch.

In [ ]:
device = get_device(None)
log_environment_info(device)

print('\nExecuting GPU Sanity Smoke Test:')
torch.manual_seed(42)

# 1. Instantiate Model & Move to Device
model = MultiOutputSeq2Point.from_config(repo_root / 'ML/configs/model_seq2point.yaml')
model.to(device)
model.train()

# 2. Synthetic Mini-Batch on Device
batch_size = 4
X_synth = torch.randn(batch_size, 599, 1, device=device, dtype=torch.float32)
y_synth = torch.randn(batch_size, 3, device=device, dtype=torch.float32)
mask_synth = torch.ones(batch_size, 3, device=device, dtype=torch.bool)

# 3. Forward Pass
preds = model(X_synth)
print(f'  - Forward Output Shape: {preds.shape} [Device: {preds.device}]')
assert preds.shape == (batch_size, 3)
assert not torch.isnan(preds).any()

# 4. Loss Computation
criterion = MaskedMultiTaskLoss()
loss, components = criterion(preds, y_synth, mask_synth, return_components=True)
print(f'  - Masked Loss Value   : {loss.item():.4f}')
assert torch.isfinite(loss)

# 5. Backward Pass
loss.backward()
for name, param in model.named_parameters():
    assert param.grad is not None
    assert not torch.isnan(param.grad).any()

print(f'  [PASS] GPU Forward/Backward Smoke Test Successful on {device}!')

### 9. GPU Performance & Throughput Benchmark
Measures inference and backpropagation throughput (batches/sec, samples/sec, ms/batch) on the active device.

In [ ]:
import time

benchmark_bsz = 64
num_warmup = 10
num_steps = 50

print(f'Running Throughput Benchmark on {device} (Batch Size = {benchmark_bsz})...')
X_bench = torch.randn(benchmark_bsz, 599, 1, device=device)
y_bench = torch.randn(benchmark_bsz, 3, device=device)
mask_bench = torch.ones(benchmark_bsz, 3, device=device, dtype=torch.bool)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Warmup
for _ in range(num_warmup):
    optimizer.zero_grad()
    p = model(X_bench)
    l = criterion(p, y_bench, mask_bench)
    l.backward()
    optimizer.step()

if device.type == 'cuda':
    torch.cuda.synchronize()

t0 = time.time()
for _ in range(num_steps):
    optimizer.zero_grad()
    p = model(X_bench)
    l = criterion(p, y_bench, mask_bench)
    l.backward()
    optimizer.step()

if device.type == 'cuda':
    torch.cuda.synchronize()

elapsed = time.time() - t0
ms_per_batch = (elapsed / num_steps) * 1000
samples_per_sec = (num_steps * benchmark_bsz) / elapsed

print(f'Benchmark Results:')
print(f'  - Latency     : {ms_per_batch:.2f} ms / batch')
print(f'  - Throughput  : {samples_per_sec:,.1f} samples / second')
print(f'  - Estimated Epoch Time (1.14M windows): {(1144163 / samples_per_sec) / 60:.2f} minutes / epoch')

### 10. Authoritative Milestone 4.5 Full Baseline Training Run

> [!CAUTION]
> **SAFETY GUARD:** Full training is **LOCKED BY DEFAULT**.
> To execute the authoritative $1,144,163$-window training run, manually change `RUN_FULL_TRAINING = False` to `RUN_FULL_TRAINING = True`.

In [ ]:
# ==============================================================================
# SAFETY LOCK: SET TO TRUE ONLY FOR THE AUTHORITATIVE TRAINING RUN
# ==============================================================================
RUN_FULL_TRAINING = False

if not RUN_FULL_TRAINING:
    print('=' * 80)
    print('[SAFETY LOCK ACTIVE] Full M4.5 training execution is LOCKED.')
    print('Set RUN_FULL_TRAINING = True when you are ready to execute the full training run.')
    print('=' * 80)
else:
    from torch.utils.data import DataLoader
    import yaml

    print('Starting Authoritative M4.5 Training Run on CUDA/GPU...')
    
    # 1. Shard Paths
    train_shards = [processed_dir / f'train/house_{h}.npz' for h in [2, 3, 5, 7, 9]]
    val_shards = [processed_dir / f'validation/house_{h}.npz' for h in [1, 8, 15]]
    
    # 2. Datasets & Loaders
    train_ds = REFITShardDataset(train_shards)
    val_ds = REFITShardDataset(val_shards)
    
    loader_kwargs = get_dataloader_kwargs(device)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, **loader_kwargs)
    
    # 3. Model, Loss, Optimizer, Scheduler
    model = MultiOutputSeq2Point.from_config(repo_root / 'ML/configs/model_seq2point.yaml')
    criterion = MaskedMultiTaskLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, min_lr=1e-6)
    
    # 4. Normalization Stats & Evaluator
    with open(repo_root / 'ML/configs/refit_normalization_stats.yaml', 'r') as f:
        norm_stats = yaml.safe_load(f)['channel_statistics']
    evaluator = Evaluator(criterion, normalization_stats=norm_stats, device=device)
    
    # 5. Checkpoint & History
    exp_dir = repo_root / 'ML/results/experiments/m4_5_seq2point_baseline'
    exp_dir.mkdir(parents=True, exist_ok=True)
    ckpt_mgr = CheckpointManager(exp_dir, monitor='val_loss', mode='min')
    history = TrainingHistory(exp_dir)
    
    # 6. Trainer Execution
    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        criterion=criterion,
        train_loader=train_loader,
        val_loader=val_loader,
        evaluator=evaluator,
        checkpoint_manager=ckpt_mgr,
        history=history,
        scheduler=scheduler,
        device=device,
        grad_clip_norm=5.0,
        early_stopping_patience=3,
    )
    
    results = trainer.fit(max_epochs=10)

### 11. Artifact Persistence & Google Drive Backup
Archive and persist trained checkpoints, training logs, loss curves, and evaluation metrics to Google Drive.

In [ ]:
print('Artifact Persistence Directory:')
exp_dir = repo_root / 'ML/results/experiments/m4_5_seq2point_baseline'
print(f'  Source: {exp_dir}')

# Example backup command once training is complete:
# !zip -r /content/drive/MyDrive/m4_5_seq2point_baseline.zip ML/results/experiments/m4_5_seq2point_baseline/